In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [15]:
mongodb=pd.read_json('../../output/mongodb/query_performance_stats.json',orient='records').set_index('query_name')
hadoop=pd.read_csv('../../output/hadoop/MR_performance_stats.csv',index_col='query_name')

# -- order alphabetically ---
mongodb=mongodb.sort_index()
hadoop=hadoop.sort_index()

# remove start_end_date_incidence_stats from mongodb
mongodb=mongodb.drop(index='country_daily_stats_USA')
mongodb=mongodb.drop(index='start_end_date_incidence_stats')
hadoop=hadoop.drop(index='start_end_date_incidence_stats')
mongodb

,_id,executionTimeMillis,totalDocsExamined,nReturned,timestamp
query_name,,,,,
continent_stats,6911a4a23a747ca371ce5f47,66.0,61900.0,6.0,2025-11-10 08:38:58.773000+00:00
country_stats,6911a4a23a747ca371ce5f48,127.0,61900.0,61900.0,2025-11-10 08:38:58.918000+00:00
weekly_stats,6911a4a33a747ca371ce5f4b,117.0,61900.0,51.0,2025-11-10 08:38:59.913000+00:00
worldwide_daily_stats,6911a4a33a747ca371ce5f4a,122.0,61900.0,61900.0,2025-11-10 08:38:59.653000+00:00


In [4]:
hadoop.columns,mongodb.columns

(Index(['mapInputRecords', 'reduceOutputRecords', 'jobDurationMillis',
        'physicalMemoryBytes', 'virtualMemoryBytes', 'cpuTimeMillis',
        'launchedMapTasks', 'launchedReduceTasks', 'bytesReadHDFS',
        'bytesWrittenHDFS'],
       dtype='object'),
 Index(['_id', 'executionTimeMillis', 'totalDocsExamined', 'nReturned',
        'timestamp'],
       dtype='object'))

In [10]:
mongodb['executionTimeMillis']

query_name
continent_stats                    66.0
country_stats                     127.0
start_end_date_incidence_stats      NaN
weekly_stats                      117.0
worldwide_daily_stats             122.0
Name: executionTimeMillis, dtype: float64

In [16]:
# - line plot comparing execution_time (executionTimeMillis,cpuTimeMillis) between mongodb and hadoop for each query

fig = go.Figure()
fig.add_trace(go.Line(x=mongodb.index, y=mongodb['executionTimeMillis'], name='MongoDB Execution Time (ms)'))
fig.add_trace(go.Line(x=hadoop.index, y=hadoop['cpuTimeMillis'], name='Hadoop Execution Time (ms)'))

fig.update_layout(title='Execution Time Comparison between MongoDB and Hadoop',
                   xaxis_title='Query Name',
                   yaxis_title='Execution Time (ms)')
fig.show()

/home/raysas/miniconda3/lib/python3.13/site-packages/plotly/graph_objs/_deprecations.py:378: DeprecationWarning:

plotly.graph_objs.Line is deprecated.
Please replace it with one of the following more specific types
  - plotly.graph_objs.scatter.Line
  - plotly.graph_objs.layout.shape.Line
  - etc.




In [18]:
fig=go.Figure()
fig.add_trace(go.Bar(x=mongodb.index, y=mongodb['totalDocsExamined'], name='MongoDB Documents Examined'))
fig.add_trace(go.Bar(x=hadoop.index, y=hadoop['mapInputRecords'], name='Hadoop MapInput Records'))
fig.update_layout(barmode='group',
                   title='Documents/Records Processed Comparison between MongoDB and Hadoop',
                   xaxis_title='Query Name',
                   yaxis_title='Number of Documents/Records Processed')
fig.show()

In [30]:
hadoop.columns

Index(['mapInputRecords', 'reduceOutputRecords', 'jobDurationMillis',
       'physicalMemoryBytes', 'virtualMemoryBytes', 'cpuTimeMillis',
       'launchedMapTasks', 'launchedReduceTasks', 'bytesReadHDFS',
       'bytesWrittenHDFS'],
      dtype='object')

In [38]:
df=hadoop
df['query_name']=df.index

# -- remove country_stats
df=df.drop(index='country_stats')

# 1️⃣ Physical vs Virtual Memory per Job
# -----------------------------
fig1 = px.bar(
    df,
    x='query_name',
    y=['physicalMemoryBytes', 'virtualMemoryBytes'],
    title='Physical vs Virtual Memory per Job',
    # barmode='group',
    labels={'value': 'Memory (Bytes)', 'query_name': 'Job / Query'}
)
fig1.show()

# -----------------------------
# 2️⃣ Memory vs Execution Time
# -----------------------------
fig2 = px.scatter(
    df,
    x='jobDurationMillis',
    y='physicalMemoryBytes',
    text='query_name',
    title='Physical Memory vs Job Duration',
    labels={'jobDurationMillis': 'Job Duration (ms)', 'physicalMemoryBytes': 'Physical Memory (Bytes)'},
    hover_data=['virtualMemoryBytes', 'cpuTimeMillis']
)
fig2.update_traces(marker=dict(size=12, color='blue', opacity=0.7))
fig2.show()

# -----------------------------
# 3️⃣ Memory vs Data Processed (input records)
# -----------------------------
fig3 = px.scatter(
    df,
    x='mapInputRecords',
    y='physicalMemoryBytes',
    size='reduceOutputRecords',  # bubble size = output records
    color='cpuTimeMillis',       # color = CPU time
    hover_name='query_name',
    title='Memory vs Input Records (Bubble size = reduce output, color = CPU time)',
    labels={'mapInputRecords': 'Map Input Records', 'physicalMemoryBytes': 'Physical Memory (Bytes)'}
)
fig3.show()

# -----------------------------
# 4️⃣ Memory Distribution
# -----------------------------
fig4 = px.histogram(
    df,
    x='physicalMemoryBytes',
    nbins=20,
    title='Distribution of Physical Memory Usage Across Jobs',
    labels={'physicalMemoryBytes': 'Physical Memory (Bytes)'}
)
fig4.show()

# -----------------------------
# 5️⃣ Memory vs CPU Usage
# -----------------------------
fig5 = px.scatter(
    df,
    x='cpuTimeMillis',
    y='physicalMemoryBytes',
    text='query_name',
    title='Physical Memory vs CPU Time',
    labels={'cpuTimeMillis': 'CPU Time (ms)', 'physicalMemoryBytes': 'Physical Memory (Bytes)'}
)
fig5.update_traces(marker=dict(size=12, color='green', opacity=0.7))
fig5.show()


In [19]:
fig=go.Figure()
fig.add_trace(go.Bar(x=mongodb.index, y=mongodb['nReturned'], name='MongoDB Documents Returned'))
fig.add_trace(go.Bar(x=hadoop.index, y=hadoop['reduceOutputRecords'], name='Hadoop ReduceOutput Records'))
fig.update_layout(barmode='group',
                   title='Documents/Records Returned Comparison between MongoDB and Hadoop',
                   xaxis_title='Query Name',
                   yaxis_title='Number of Documents/Records Returned')